final final

In [1]:
# final for pm and dense granules.
import cv2
import numpy as np
import os

def analyze_platelet(image_path):
    """
    Analyze platelet in an image with interactive point selection and dual threshold adjustment.
    
    Args:
        image_path (str): Path to the image file
    
    Returns:
        dict: Analysis results containing contour count, areas, and file paths
    """
    
    # --- Load image ---
    image = cv2.imread(image_path)
    if image is None:
        raise ValueError(f"Could not load image from {image_path}")
    
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
    original_height, original_width = gray.shape

    # --- Global variables ---
    threshold_value_1 = 135  # First threshold
    threshold_value_2 = 120  # Second threshold
    zoom_factor = 1.0
    zoom_step = 0.1
    min_zoom, max_zoom = 0.2, 4.0
    pan_offset = [0, 0]
    drag_start = None
    clicked_points = []
    current_display = None
    current_binary_1 = None
    current_binary_2 = None
    phase = 1  # 1 = point selection, 2 = first threshold, 3 = second threshold
    mask = None
    window_width = 1200
    window_height = 800
    min_contour_area = 1000  # default minimum area
    max_contour_area = 800000  # default max area
    trackbars_initialized = False  # Flag to prevent early trackbar access
    
    # Store contours from both thresholds
    first_threshold_contours = []
    second_threshold_contours = []
    updating_display = False  # Flag to prevent callback conflicts

    def nothing(x):
        pass

    def update_threshold_1():
        """Update first binary image based on current threshold"""
        nonlocal current_binary_1
        _, current_binary_1 = cv2.threshold(gray, threshold_value_1, 255, cv2.THRESH_BINARY_INV)

    def update_threshold_2():
        """Update second binary image based on current threshold"""
        nonlocal current_binary_2
        _, current_binary_2 = cv2.threshold(gray, threshold_value_2, 255, cv2.THRESH_BINARY_INV)

    def get_display_image():
        """Get the properly zoomed and panned image"""
        # Calculate display dimensions based on zoom
        display_w = int(original_width * zoom_factor)
        display_h = int(original_height * zoom_factor)
        
        # Resize the image based on zoom factor
        zoomed_image = cv2.resize(image, (display_w, display_h), interpolation=cv2.INTER_LINEAR)
        
        # Apply pan offset - create a window view
        start_x = max(0, pan_offset[0])
        start_y = max(0, pan_offset[1])
        end_x = min(display_w, start_x + window_width)
        end_y = min(display_h, start_y + window_height)
        
        # Extract the visible portion
        if start_x < display_w and start_y < display_h:
            cropped = zoomed_image[start_y:end_y, start_x:end_x].copy()
        else:
            cropped = np.zeros((window_height, window_width, 3), dtype=np.uint8)
        
        return cropped, start_x, start_y

    def draw_contours_on_overlay(overlay, contours, color, view_start_x, view_start_y):
        """Helper function to draw contours on overlay with proper transformation"""
        for cnt in contours:
            # Transform contour points for current view
            transformed_cnt = []
            for point in cnt:
                x, y = point[0]
                # Scale to zoom level
                display_x = int(x * zoom_factor) - view_start_x
                display_y = int(y * zoom_factor) - view_start_y
                
                if 0 <= display_x < overlay.shape[1] and 0 <= display_y < overlay.shape[0]:
                    transformed_cnt.append([[display_x, display_y]])
            
            if transformed_cnt:
                transformed_cnt = np.array(transformed_cnt, dtype=np.int32)
                cv2.drawContours(overlay, [transformed_cnt], -1, color, 2)

    def update_display():
        """Update the display with current zoom, pan, and overlay"""
        nonlocal current_display
        
        # Get the display image with proper zoom and pan
        display_img, view_start_x, view_start_y = get_display_image()
        overlay = display_img.copy()
        
        # In phase 2, show first threshold contours only within the selected mask
        if phase == 2 and current_binary_1 is not None and mask is not None:
            # Create masked binary for contour detection
            masked_binary = cv2.bitwise_and(current_binary_1, current_binary_1, mask=mask)
            contours, _ = cv2.findContours(masked_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            
            # Filter contours by area
            valid_contours = [cnt for cnt in contours if min_contour_area < cv2.contourArea(cnt) < max_contour_area]
            
            if not valid_contours:
                # Use the polygon itself as the contour
                polygon_contour = np.array(clicked_points, dtype=np.int32).reshape(-1, 1, 2)
                valid_contours = [polygon_contour]
            
            # Draw first threshold contours in green
            draw_contours_on_overlay(overlay, valid_contours, (0, 255, 0), view_start_x, view_start_y)
        
        # In phase 3, show both threshold contours
        elif phase == 3 and mask is not None:
            # Draw saved first threshold contours in green
            draw_contours_on_overlay(overlay, first_threshold_contours, (0, 255, 0), view_start_x, view_start_y)
            
            # Draw current second threshold contours in blue
            if current_binary_2 is not None:
                masked_binary = cv2.bitwise_and(current_binary_2, current_binary_2, mask=mask)
                contours, _ = cv2.findContours(masked_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                # Filter contours by area
                valid_contours = [cnt for cnt in contours if min_contour_area < cv2.contourArea(cnt) < max_contour_area]
                
                if not valid_contours:
                    # Use the polygon itself as the contour
                    polygon_contour = np.array(clicked_points, dtype=np.int32).reshape(-1, 1, 2)
                    valid_contours = [polygon_contour]
                
                # Draw second threshold contours in blue
                draw_contours_on_overlay(overlay, valid_contours, (255, 0, 0), view_start_x, view_start_y)
        
        # Draw clicked points and lines
        for i, pt in enumerate(clicked_points):
            # Transform point to display coordinates
            display_x = int(pt[0] * zoom_factor) - view_start_x
            display_y = int(pt[1] * zoom_factor) - view_start_y
            
            if 0 <= display_x < overlay.shape[1] and 0 <= display_y < overlay.shape[0]:
                cv2.circle(overlay, (display_x, display_y), 3, (0, 255, 0), -1)
                # Add point number
                cv2.putText(overlay, str(i+1), (display_x+5, display_y-5), 
                           cv2.FONT_HERSHEY_SIMPLEX, 0.4, (0, 255, 0), 1)
        
        # Draw lines between consecutive points
        if len(clicked_points) > 1:
            for i in range(len(clicked_points) - 1):
                pt1_x = int(clicked_points[i][0] * zoom_factor) - view_start_x
                pt1_y = int(clicked_points[i][1] * zoom_factor) - view_start_y
                pt2_x = int(clicked_points[i+1][0] * zoom_factor) - view_start_x
                pt2_y = int(clicked_points[i+1][1] * zoom_factor) - view_start_y
                
                if (0 <= pt1_x < overlay.shape[1] and 0 <= pt1_y < overlay.shape[0] and
                    0 <= pt2_x < overlay.shape[1] and 0 <= pt2_y < overlay.shape[0]):
                    cv2.line(overlay, (pt1_x, pt1_y), (pt2_x, pt2_y), (0, 255, 0), 2)
            
            # Close polygon if we have enough points
            if len(clicked_points) >= 3:
                pt1_x = int(clicked_points[-1][0] * zoom_factor) - view_start_x
                pt1_y = int(clicked_points[-1][1] * zoom_factor) - view_start_y
                pt2_x = int(clicked_points[0][0] * zoom_factor) - view_start_x
                pt2_y = int(clicked_points[0][1] * zoom_factor) - view_start_y
                
                if (0 <= pt1_x < overlay.shape[1] and 0 <= pt1_y < overlay.shape[0] and
                    0 <= pt2_x < overlay.shape[1] and 0 <= pt2_y < overlay.shape[0]):
                    cv2.line(overlay, (pt1_x, pt1_y), (pt2_x, pt2_y), (0, 255, 0), 2)
        
        # Add status text based on phase
        if phase == 1:
            text = f"PHASE 1 - Select Points | Zoom: {zoom_factor:.1f}x | Points: {len(clicked_points)}/3 min"
            color = (255, 255, 0)  # Yellow
        elif phase == 2:
            text = f"PHASE 2 - First Threshold: {threshold_value_1} (GREEN) | Area: {min_contour_area}-{max_contour_area} | Zoom: {zoom_factor:.1f}x"
            color = (0, 255, 0)  # Green
        else:  # phase == 3
            text = f"PHASE 3 - Second Threshold: {threshold_value_2} (BLUE) | GREEN: {threshold_value_1} | Zoom: {zoom_factor:.1f}x"
            color = (255, 0, 0)  # Blue (BGR format)
        
        cv2.putText(overlay, text, (10, 25), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)
        
        # Add instructions
        if phase == 1:
            instructions = "Left-click: Add point | Right-drag: Pan | +/-: Zoom | u: Undo | t: Next phase"
        elif phase == 2:
            instructions = "Trackbar: Adjust threshold | n: Save & next threshold | s: Skip to final"
        else:  # phase == 3
            instructions = "Trackbar: Adjust second threshold | s: Save final result"
        cv2.putText(overlay, instructions, (10, 50), cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255, 255, 255), 1)
        
        current_display = overlay
        return overlay

    def click_event(event, x, y, flags, param):
        """Handle mouse events for clicking points and panning"""
        nonlocal current_display, drag_start, pan_offset
        
        if phase == 1 and event == cv2.EVENT_LBUTTONDOWN:
            # Convert display coordinates back to original image coordinates
            # Account for pan and zoom
            view_start_x = max(0, pan_offset[0])
            view_start_y = max(0, pan_offset[1])
            
            orig_x = int((x + view_start_x) / zoom_factor)
            orig_y = int((y + view_start_y) / zoom_factor)
            
            # Ensure coordinates are within bounds
            orig_x = max(0, min(orig_x, original_width - 1))
            orig_y = max(0, min(orig_y, original_height - 1))
            
            clicked_points.append((orig_x, orig_y))
            print(f"Point {len(clicked_points)}: ({orig_x}, {orig_y})")
            
            current_display = update_display()
            cv2.imshow("Platelet Analyzer", current_display)
        
        elif event == cv2.EVENT_RBUTTONDOWN:
            drag_start = (x, y)
        
        elif event == cv2.EVENT_MOUSEMOVE and drag_start:
            dx = x - drag_start[0]
            dy = y - drag_start[1]
            
            # Update pan offset
            max_pan_x = max(0, int(original_width * zoom_factor) - window_width)
            max_pan_y = max(0, int(original_height * zoom_factor) - window_height)
            
            pan_offset[0] = max(0, min(pan_offset[0] - dx, max_pan_x))
            pan_offset[1] = max(0, min(pan_offset[1] - dy, max_pan_y))
            
            drag_start = (x, y)
            current_display = update_display()
            cv2.imshow("Platelet Analyzer", current_display)
        
        elif event == cv2.EVENT_RBUTTONUP:
            drag_start = None

    def trackbar_callback(val):
        """Callback for trackbar changes - only active when trackbars are initialized"""
        nonlocal threshold_value_1, threshold_value_2, min_contour_area, max_contour_area, updating_display
        
        # Only proceed if trackbars are properly initialized and not already updating
        if not trackbars_initialized or updating_display:
            return
            
        updating_display = True
        try:
            if phase == 2:
                threshold_value_1 = max(10, min(cv2.getTrackbarPos("Threshold", "Platelet Analyzer") * 5, 250))
                update_threshold_1()
            elif phase == 3:
                threshold_value_2 = max(10, min(cv2.getTrackbarPos("Threshold", "Platelet Analyzer") * 5, 250))
                update_threshold_2()
            
            min_contour_area = max(100, cv2.getTrackbarPos("Min Area", "Platelet Analyzer") * 100)
            max_contour_area = max(min_contour_area + 100, cv2.getTrackbarPos("Max Area", "Platelet Analyzer") * 100)

            if phase >= 2:
                current_display = update_display()
                cv2.imshow("Platelet Analyzer", current_display)
        except cv2.error as e:
            print(f"Trackbar error (ignoring): {e}")
        finally:
            updating_display = False

    # --- Initialize ---
    print("=== Dual Threshold Platelet Analysis Tool ===")
    print("Phase 1: Select points around platelet boundary")
    print("Phase 2: Adjust first threshold (GREEN contours)")
    print("Phase 3: Adjust second threshold (BLUE contours)")
    print("\nControls:")
    print("- Left-click: Select points (Phase 1 only)")
    print("- Trackbar: Adjust threshold (Phase 2 & 3)")
    print("- Right-click + drag: Pan view")
    print("- '+'/'-': Zoom in/out")
    print("- 'u': Undo last point (Phase 1 only)")
    print("- 't': Move to first threshold phase")
    print("- 'n': Save current threshold and move to second threshold")
    print("- 's': Save and process selection")
    print("- ESC: Exit")

    # STEP 1: Create window first
    cv2.namedWindow("Platelet Analyzer", cv2.WINDOW_NORMAL)
    cv2.resizeWindow("Platelet Analyzer", window_width, window_height)

    # STEP 2: Set mouse callback
    cv2.setMouseCallback("Platelet Analyzer", click_event)

    # STEP 3: Initialize threshold and display BEFORE creating trackbars
    update_threshold_1()
    update_threshold_2()
    current_display = update_display()
    cv2.imshow("Platelet Analyzer", current_display)

    # STEP 4: Create trackbars with initial values (this will trigger callbacks)
    cv2.createTrackbar("Threshold", "Platelet Analyzer", 65, 125, trackbar_callback)  # 27*5 = 160
    cv2.createTrackbar("Min Area", "Platelet Analyzer", 60, 500, trackbar_callback)    # 60*100 = 1000
    cv2.createTrackbar("Max Area", "Platelet Analyzer", 80000, 800000, trackbar_callback) # 80000*100 = 800000

    # STEP 5: Now mark trackbars as initialized
    trackbars_initialized = True
    
    # STEP 6: Update values from trackbars now that they're initialized
    try:
        threshold_value_1 = max(10, min(cv2.getTrackbarPos("Threshold", "Platelet Analyzer") * 2, 250))
        min_contour_area = max(100, cv2.getTrackbarPos("Min Area", "Platelet Analyzer") * 100)
        max_contour_area = max(min_contour_area + 100, cv2.getTrackbarPos("Max Area", "Platelet Analyzer") * 100)
    except cv2.error:
        pass  # Use defaults if reading fails

    # --- Main interaction loop ---
    print(f"\n=== PHASE {phase}: POINT SELECTION ===")
    print("Click points around the platelet boundary (minimum 3 points)")
    print("Use zoom/pan controls to navigate. Press 't' when done selecting.")

    while True:
        key = cv2.waitKey(30) & 0xFF
        
        if key == 27:  # ESC - Exit
            cv2.destroyAllWindows()
            return {"status": "cancelled", "message": "Analysis cancelled by user"}
        
        elif key == ord('t'):  # Move to first threshold phase
            if phase == 1:
                if len(clicked_points) >= 3:
                    # Create mask from selected points
                    mask = np.zeros((original_height, original_width), dtype=np.uint8)
                    polygon = np.array([clicked_points], dtype=np.int32)
                    cv2.fillPoly(mask, [polygon], 255)
                    
                    phase = 2
                    print(f"\n=== PHASE {phase}: FIRST THRESHOLD ADJUSTMENT (GREEN) ===")
                    print("Adjust the first threshold using the trackbar")
                    print("Press 'n' to save this threshold and move to second threshold")
                    print("Press 's' to skip second threshold and save final result")
                    
                    # Set trackbar to current threshold value
                    cv2.setTrackbarPos("Threshold", "Platelet Analyzer", threshold_value_1 // 5)
                    
                    # Update display to show threshold effects within mask
                    update_threshold_1()
                    current_display = update_display()
                    cv2.imshow("Platelet Analyzer", current_display)
                else:
                    print("Need at least 3 points to create mask. Continue selecting points.")
            else:
                print("Already past phase 1.")
        
        elif key == ord('n'):  # Save current threshold and move to next
            if phase == 2:
                # Save first threshold contours
                masked_binary = cv2.bitwise_and(current_binary_1, current_binary_1, mask=mask)
                contours, _ = cv2.findContours(masked_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                
                # Filter contours by area
                valid_contours = [cnt for cnt in contours if min_contour_area < cv2.contourArea(cnt) < max_contour_area]
                
                if not valid_contours:
                    # Use the polygon itself as the contour
                    polygon_contour = np.array(clicked_points, dtype=np.int32).reshape(-1, 1, 2)
                    valid_contours = [polygon_contour]
                
                first_threshold_contours = valid_contours.copy()
                print(f"Saved {len(first_threshold_contours)} contours from first threshold ({threshold_value_1})")
                
                # Move to phase 3
                phase = 3
                print(f"\n=== PHASE {phase}: SECOND THRESHOLD ADJUSTMENT (BLUE) ===")
                print("Adjust the second threshold using the trackbar")
                print("First threshold contours are shown in GREEN")
                print("Second threshold contours are shown in BLUE")
                print("Press 's' when satisfied with both results")
                
                # Set trackbar to second threshold value
                cv2.setTrackbarPos("Threshold", "Platelet Analyzer", threshold_value_2 // 5)
                
                # Update display
                update_threshold_2()
                current_display = update_display()
                cv2.imshow("Platelet Analyzer", current_display)
            else:
                print("Only available in phase 2.")
        
        elif key == ord('+') or key == ord('='):  # Zoom in
            zoom_factor = min(zoom_factor + zoom_step, max_zoom)
            current_display = update_display()
            cv2.imshow("Platelet Analyzer", current_display)
        
        elif key == ord('-') or key == ord('_'):  # Zoom out
            zoom_factor = max(zoom_factor - zoom_step, min_zoom)
            current_display = update_display()
            cv2.imshow("Platelet Analyzer", current_display)
        
        elif key == ord('u') and phase == 1:  # Undo last point (only in phase 1)
            if clicked_points:
                removed_point = clicked_points.pop()
                print(f"Removed point: {removed_point}")
                current_display = update_display()
                cv2.imshow("Platelet Analyzer", current_display)
        
        elif key == ord('s'):  # Save and process
            if phase >= 2 and len(clicked_points) >= 3:
                if phase == 2:
                    # Save first threshold contours if not already saved
                    masked_binary = cv2.bitwise_and(current_binary_1, current_binary_1, mask=mask)
                    contours, _ = cv2.findContours(masked_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    valid_contours = [cnt for cnt in contours if min_contour_area < cv2.contourArea(cnt) < max_contour_area]
                    
                    if not valid_contours:
                        polygon_contour = np.array(clicked_points, dtype=np.int32).reshape(-1, 1, 2)
                        valid_contours = [polygon_contour]
                    
                    first_threshold_contours = valid_contours.copy()
                    print(f"Using single threshold with {len(first_threshold_contours)} contours")
                
                elif phase == 3:
                    # Save second threshold contours
                    masked_binary = cv2.bitwise_and(current_binary_2, current_binary_2, mask=mask)
                    contours, _ = cv2.findContours(masked_binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
                    
                    valid_contours = [cnt for cnt in contours if min_contour_area < cv2.contourArea(cnt) < max_contour_area]
                    
                    if not valid_contours:
                        polygon_contour = np.array(clicked_points, dtype=np.int32).reshape(-1, 1, 2)
                        valid_contours = [polygon_contour]
                    
                    second_threshold_contours = valid_contours.copy()
                    print(f"Using dual thresholds: {len(first_threshold_contours)} (GREEN) + {len(second_threshold_contours)} (BLUE) contours")
                
                print(f"\n=== PROCESSING COMPLETE ===")
                break
            elif phase == 1:
                print("Complete point selection first, then press 't' to move to threshold adjustment")
            else:
                print("Need at least 3 points and to be in threshold phase.")

    cv2.destroyAllWindows()

    # --- Process and save results ---
    if len(clicked_points) >= 3 and mask is not None:
        print("Processing and saving results...")
        
        # Save debug images
        cv2.imwrite("debug_polygon_mask.png", mask)
        print("Saved: debug_polygon_mask.png")
        
        # Create final output image
        output = image.copy()
        
        # Draw first threshold contours in green
        total_contours = 0
        all_areas = []
        
        if first_threshold_contours:
            for cnt in first_threshold_contours:
                area = cv2.contourArea(cnt)
                cv2.drawContours(output, [cnt], -1, (0, 255, 0), 2)  # Green
                total_contours += 1
                all_areas.append(("First", area, threshold_value_1))
                print(f"First Threshold Contour {total_contours}: Area = {area:.0f} pixels (threshold: {threshold_value_1})")
        
        # Draw second threshold contours in blue (if exists)
        if second_threshold_contours:
            for cnt in second_threshold_contours:
                area = cv2.contourArea(cnt)
                cv2.drawContours(output, [cnt], -1, (255, 0, 0), 2)  # Blue
                total_contours += 1
                all_areas.append(("Second", area, threshold_value_2))
                print(f"Second Threshold Contour {total_contours}: Area = {area:.0f} pixels (threshold: {threshold_value_2})")
        

        # Save final result
        cv2.imwrite("platelet_dual_threshold_final.png", output)
        print("Saved: platelet_dual_threshold_final.png")
        
        # Display final result
        scale_percent = 30
        display_width = int(original_width * scale_percent / 100)
        display_height = int(original_height * scale_percent / 100)
        resized_output = cv2.resize(output, (display_width, display_height))
        
        cv2.imshow("Final Dual Threshold Result", resized_output)
        print(f"\n=== DUAL THRESHOLD ANALYSIS COMPLETE ===")
        print(f"Total contours: {total_contours}")
        print(f"First threshold: {threshold_value_1} (GREEN)")
        if second_threshold_contours:
            print(f"Second threshold: {threshold_value_2} (BLUE)")
        print("Press any key to exit...")
        cv2.waitKey(5000)
        cv2.destroyAllWindows()
        
        # Return results
        return {
            "input_image": image_path,
            "status": "completed",
            "total_contour_count": total_contours,
            "first_threshold": {
                "value": threshold_value_1,
                "contour_count": len(first_threshold_contours),
                "areas": [area for _, area, _ in all_areas if _ == "First"]
            },
            "second_threshold": {
                "value": threshold_value_2,
                "contour_count": len(second_threshold_contours),
                "areas": [area for _, area, _ in all_areas if _ == "Second"]
            } if second_threshold_contours else None,
            "all_contour_details": all_areas,
            "min_contour_area": min_contour_area,
            "max_contour_area": max_contour_area,
            "selected_points": clicked_points,
            "output_files": {
                "mask": "debug_polygon_mask.png",
                "final_result": "platelet_dual_threshold_final.png"
            }
        }
    
    else:
        return {"status": "cancelled", "message": "Analysis cancelled - need completed selection and threshold adjustment."}

In [2]:
import cv2
import numpy as np
import os

def mark_oc_with_gui(image_path,output_folder, output_file_path):
    """
    Interactive detection of alpha granules using HSV + white thresholding.
    Includes dynamic threshold and area filtering.
    """

    image = cv2.imread(image_path)
    if image is None:
        print(f"Error: Could not read image at {image_path}")
        return

    hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

    # Create window and trackbars
    window_name = 'Open Canalicular Detection'
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    def nothing(x): pass

    cv2.createTrackbar('White Thresh', window_name, 35, 135, nothing)  # 0-135 maps to 120-255
    cv2.createTrackbar('Min Area', window_name, 7000, 100000, nothing)
    cv2.createTrackbar('Max Area', window_name, 1000000, 1000000, nothing)

    # Define green mask
    lower_green = np.array([40, 50, 50])
    upper_green = np.array([80, 255, 255])
    green_mask = cv2.inRange(hsv, lower_green, upper_green)

    # Fill region inside green contour
    contours, _ = cv2.findContours(green_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    filled_mask = np.zeros_like(green_mask)
    cv2.drawContours(filled_mask, contours, -1, 255, thickness=cv2.FILLED)

    masked_image = cv2.bitwise_and(image, image, mask=filled_mask)
    gray = cv2.cvtColor(masked_image, cv2.COLOR_BGR2GRAY)

    while True:
        # Read trackbar values
        white_thresh_raw = cv2.getTrackbarPos('White Thresh', window_name)
        white_thresh = white_thresh_raw + 120  # Map 0-135 to 120-255
        min_area = cv2.getTrackbarPos('Min Area', window_name)
        max_area = cv2.getTrackbarPos('Max Area', window_name)

        # Threshold bright regions
        _, white_mask = cv2.threshold(gray, white_thresh, 255, cv2.THRESH_BINARY)

        # Find and filter contours
        white_contours, _ = cv2.findContours(white_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        filtered_contours = [cnt for cnt in white_contours if min_area < cv2.contourArea(cnt) < max_area]

        # Draw filtered contours
        result = image.copy()
        cv2.drawContours(result, filtered_contours, -1, (0, 0, 255), 1)

        cv2.imshow(window_name, result)

        key = cv2.waitKey(10) & 0xFF
        if key == ord('s'):
            # out_path = os.path.join(os.path.dirname(image_path), 'platelet_alpha_granules_marked.png')
            cv2.imwrite(output_file_path, result)
            print(f"Saved: {output_file_path}")
        elif key == 27:  # ESC to exit
            break

    cv2.destroyAllWindows()


In [3]:
import os
import cv2
import numpy as np

input_folder = r"C:\Users\Sharmaji\AIIMS_platelet\trial"
for fname in os.listdir(input_folder):
    # if not fname.endswith('.png'):
    #     continue
    image_path = os.path.join(input_folder, fname)
    print(f"Processing {image_path}...")
    result = analyze_platelet(image_path)  # Call the function for each image

    if result["status"] != "completed":
        print(f"skipped {image_path}: {result['message']}")
        continue

    dual_threshold_file = r"C:/Users/Sharmaji/AIIMS_platelet/platelet_dual_threshold_final.png"

    output_folder = r"C:\Users\Sharmaji\AIIMS_platelet\trial_annotated_2"
    output_file_name = fname

    output_file_path = os.path.join(output_folder, output_file_name)
    os.makedirs(output_folder, exist_ok=True)

    mark_oc_with_gui(dual_threshold_file, output_folder, output_file_path)

    

Processing C:\Users\Sharmaji\AIIMS_platelet\trial\24-107c_16.jpg...
=== Dual Threshold Platelet Analysis Tool ===
Phase 1: Select points around platelet boundary
Phase 2: Adjust first threshold (GREEN contours)
Phase 3: Adjust second threshold (BLUE contours)

Controls:
- Left-click: Select points (Phase 1 only)
- Trackbar: Adjust threshold (Phase 2 & 3)
- Right-click + drag: Pan view
- '+'/'-': Zoom in/out
- 'u': Undo last point (Phase 1 only)
- 't': Move to first threshold phase
- 'n': Save current threshold and move to second threshold
- 's': Save and process selection
- ESC: Exit

=== PHASE 1: POINT SELECTION ===
Click points around the platelet boundary (minimum 3 points)
Use zoom/pan controls to navigate. Press 't' when done selecting.
Point 1: (740, 1095)
Point 2: (865, 1170)
Point 3: (1095, 1190)
Point 4: (1480, 730)
Point 5: (2080, 315)
Point 6: (2430, 75)
Point 7: (2655, 15)
Point 8: (3025, 0)
Point 9: (3220, 210)
Point 10: (3390, 710)
Point 11: (3625, 1080)
Point 12: (4045, 

# analysis of features

In [4]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from sklearn.cluster import DBSCAN
from scipy.spatial.distance import pdist, squareform
from scipy.spatial import distance_matrix
import pandas as pd
from collections import Counter
import seaborn as sns

class AlphaGranuleActivationAnalyzer:
    def __init__(self, image_path):
        """
        Initialize the analyzer with the annotated platelet image
        
        Args:
            image_path: Path to the annotated platelet image
        """
        self.image = cv2.imread(image_path)
        self.image_rgb = cv2.cvtColor(self.image, cv2.COLOR_BGR2RGB)
        self.alpha_granules = []
        self.centroids = []
        self.activation_metrics = {}
        
    def extract_alpha_granules(self, color_channel='blue'):
        """
        Extract alpha granule contours from the blue channel
        Assumes alpha granules are marked in blue color
        """
        # Convert to HSV for better color detection
        hsv = cv2.cvtColor(self.image, cv2.COLOR_BGR2HSV)
        
        if color_channel == 'blue':
            # Define blue color range for alpha granules
            lower_blue = np.array([100, 50, 50])
            upper_blue = np.array([130, 255, 255])
            mask = cv2.inRange(hsv, lower_blue, upper_blue)
        
        # Find contours
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        # Filter contours by area (remove noise)
        min_area = 100  # Adjust based on your image scale
        max_area = 50000  # Adjust based on your image scale
        
        filtered_contours = []
        centroids = []
        
        for contour in contours:
            area = cv2.contourArea(contour)
            if min_area < area < max_area:
                filtered_contours.append(contour)
                
                # Calculate centroid
                M = cv2.moments(contour)
                if M["m00"] != 0:
                    cx = int(M["m10"] / M["m00"])
                    cy = int(M["m01"] / M["m00"])
                    centroids.append([cx, cy])
        
        self.alpha_granules = filtered_contours
        self.centroids = np.array(centroids)
        
        print(f"Found {len(self.alpha_granules)} alpha granules")
        return len(self.alpha_granules)
    
    def calculate_distance_metrics(self):
        """Calculate various distance-based metrics"""
        if len(self.centroids) < 2:
            return {"error": "Not enough granules for analysis"}
        
        # Calculate pairwise distances
        distances = pdist(self.centroids)
        distance_matrix = squareform(distances)
        
        # Remove diagonal (distance to self = 0)
        np.fill_diagonal(distance_matrix, np.inf)
        
        # Nearest neighbor distances
        nearest_distances = np.min(distance_matrix, axis=1)
        
        # Average nearest neighbor distance
        avg_nearest_distance = np.mean(nearest_distances)
        
        # Overall statistics
        mean_distance = np.mean(distances)
        std_distance = np.std(distances)
        
        return {
            'avg_nearest_distance': avg_nearest_distance,
            'mean_all_distances': mean_distance,
            'std_distances': std_distance,
            'nearest_distances': nearest_distances,
            'distance_matrix': distance_matrix
        }
    
    def perform_clustering_analysis(self, eps_ratio=0.3):
        """
        Perform DBSCAN clustering on alpha granules
        
        Args:
            eps_ratio: Ratio of average nearest neighbor distance to use as eps
        """
        if len(self.centroids) < 2:
            return {"error": "Not enough granules for clustering"}
        
        # Calculate eps based on average nearest neighbor distance
        distance_metrics = self.calculate_distance_metrics()
        eps = distance_metrics['avg_nearest_distance'] * eps_ratio
        
        # Perform DBSCAN clustering
        clustering = DBSCAN(eps=eps, min_samples=2).fit(self.centroids)
        labels = clustering.labels_
        
        # Number of clusters (excluding noise points labeled as -1)
        n_clusters = len(set(labels)) - (1 if -1 in labels else 0)
        n_noise = list(labels).count(-1)
        
        # Calculate clustering metrics
        clustered_granules = len(labels) - n_noise
        clustering_percentage = (clustered_granules / len(labels)) * 100
        
        # Cluster sizes
        cluster_sizes = []
        if n_clusters > 0:
            cluster_counter = Counter([label for label in labels if label != -1])
            cluster_sizes = list(cluster_counter.values())
        
        return {
            'labels': labels,
            'n_clusters': n_clusters,
            'n_noise': n_noise,
            'clustered_granules': clustered_granules,
            'clustering_percentage': clustering_percentage,
            'cluster_sizes': cluster_sizes,
            'eps_used': eps
        }
    
    def calculate_proximity_analysis(self, proximity_threshold_ratio=2.0):
        """
        Analyze granules based on proximity to neighbors
        
        Args:
            proximity_threshold_ratio: Multiple of average granule size to use as threshold
        """
        if len(self.centroids) < 2:
            return {"error": "Not enough granules for analysis"}
        
        # Estimate average granule size from contour areas
        areas = [cv2.contourArea(contour) for contour in self.alpha_granules]
        avg_area = np.mean(areas)
        avg_radius = np.sqrt(avg_area / np.pi)
        
        proximity_threshold = avg_radius * proximity_threshold_ratio
        
        # Calculate distance matrix
        distance_metrics = self.calculate_distance_metrics()
        distance_matrix = distance_metrics['distance_matrix']
        
        # Count neighbors within threshold for each granule
        neighbor_counts = []
        for i in range(len(self.centroids)):
            neighbors = np.sum(distance_matrix[i] < proximity_threshold)
            neighbor_counts.append(neighbors)
        
        # Calculate metrics
        avg_neighbors = np.mean(neighbor_counts)
        isolated_granules = np.sum(np.array(neighbor_counts) == 0)
        grouped_granules = len(neighbor_counts) - isolated_granules
        grouping_percentage = (grouped_granules / len(neighbor_counts)) * 100
        
        return {
            'proximity_threshold': proximity_threshold,
            'neighbor_counts': neighbor_counts,
            'avg_neighbors': avg_neighbors,
            'isolated_granules': isolated_granules,
            'grouped_granules': grouped_granules,
            'grouping_percentage': grouping_percentage
        }
    
    def calculate_spatial_distribution_index(self):
        """
        Calculate various spatial distribution indices
        """
        if len(self.centroids) < 3:
            return {"error": "Not enough granules for spatial analysis"}
        
        # Calculate variance-to-mean ratio for x and y coordinates
        x_coords = self.centroids[:, 0]
        y_coords = self.centroids[:, 1]
        
        x_vmr = np.var(x_coords) / np.mean(x_coords) if np.mean(x_coords) > 0 else 0
        y_vmr = np.var(y_coords) / np.mean(y_coords) if np.mean(y_coords) > 0 else 0
        
        # Calculate nearest neighbor distances
        distance_metrics = self.calculate_distance_metrics()
        nearest_distances = distance_metrics['nearest_distances']
        
        # Clark-Evans index (ratio of observed to expected nearest neighbor distance)
        # Expected distance for random distribution = 0.5 / sqrt(density)
        image_area = self.image.shape[0] * self.image.shape[1]
        density = len(self.centroids) / image_area
        expected_distance = 0.5 / np.sqrt(density)
        clark_evans_index = np.mean(nearest_distances) / expected_distance
        
        return {
            'x_variance_to_mean_ratio': x_vmr,
            'y_variance_to_mean_ratio': y_vmr,
            'clark_evans_index': clark_evans_index,
            'spatial_distribution_score': (x_vmr + y_vmr) / 2
        }
    
    def classify_activation_state(self):
        """
        Classify the activation state based on multiple metrics
        """
        # Perform all analyses
        clustering_results = self.perform_clustering_analysis()
        proximity_results = self.calculate_proximity_analysis()
        spatial_results = self.calculate_spatial_distribution_index()
        distance_metrics = self.calculate_distance_metrics()
        
        if any('error' in result for result in [clustering_results, proximity_results, spatial_results]):
            return {"classification": "INSUFFICIENT_DATA", "confidence": 0.0}
        
        # Define activation criteria and scoring
        activation_score = 0
        criteria_met = []
        
        # Criterion 1: Clustering percentage (weight: 0.3)
        clustering_pct = clustering_results['clustering_percentage']
        if clustering_pct > 60:
            activation_score += 0.3
            criteria_met.append(f"High clustering: {clustering_pct:.1f}%")
        elif clustering_pct > 40:
            activation_score += 0.15
            criteria_met.append(f"Moderate clustering: {clustering_pct:.1f}%")
        
        # Criterion 2: Grouping percentage (weight: 0.25)
        grouping_pct = proximity_results['grouping_percentage']
        if grouping_pct > 70:
            activation_score += 0.25
            criteria_met.append(f"High grouping: {grouping_pct:.1f}%")
        elif grouping_pct > 50:
            activation_score += 0.125
            criteria_met.append(f"Moderate grouping: {grouping_pct:.1f}%")
        
        # Criterion 3: Average neighbors (weight: 0.2)
        avg_neighbors = proximity_results['avg_neighbors']
        if avg_neighbors > 2:
            activation_score += 0.2
            criteria_met.append(f"High neighbor count: {avg_neighbors:.1f}")
        elif avg_neighbors > 1:
            activation_score += 0.1
            criteria_met.append(f"Moderate neighbor count: {avg_neighbors:.1f}")
        
        # Criterion 4: Clark-Evans index (weight: 0.15)
        clark_evans = spatial_results['clark_evans_index']
        if clark_evans < 0.8:  # Less than random = clustered
            activation_score += 0.15
            criteria_met.append(f"Clustered distribution (CE: {clark_evans:.2f})")
        elif clark_evans < 1.0:
            activation_score += 0.075
        
        # Criterion 5: Number of clusters relative to total granules (weight: 0.1)
        n_clusters = clustering_results['n_clusters']
        n_granules = len(self.centroids)
        cluster_ratio = n_clusters / n_granules if n_granules > 0 else 1
        
        if cluster_ratio < 0.5:  # Few clusters relative to granules = more clustering
            activation_score += 0.1
            criteria_met.append(f"Few clusters relative to granules: {cluster_ratio:.2f}")
        elif cluster_ratio < 0.7:
            activation_score += 0.05
        
        # Classification based on activation score
        if activation_score >= 0.7:
            classification = "ACTIVATED"
            confidence = min(activation_score, 1.0)
        elif activation_score >= 0.4:
            classification = "PARTIALLY_ACTIVATED"
            confidence = activation_score
        else:
            classification = "UNACTIVATED"
            confidence = 1.0 - activation_score
        
        # Store comprehensive results
        self.activation_metrics = {
            'classification': classification,
            'confidence': confidence,
            'activation_score': activation_score,
            'criteria_met': criteria_met,
            'clustering_results': clustering_results,
            'proximity_results': proximity_results,
            'spatial_results': spatial_results,
            'distance_metrics': distance_metrics,
            'n_granules': n_granules
        }
        
        return self.activation_metrics
    
    def visualize_analysis(self, save_path=None):
        """
        Create comprehensive visualization of the analysis
        """
        fig, axes = plt.subplots(2, 3, figsize=(18, 12))
        fig.suptitle('Alpha Granule Activation Analysis', fontsize=16, fontweight='bold')
        
        # Original image with granules marked
        axes[0, 0].imshow(self.image_rgb)
        axes[0, 0].scatter(self.centroids[:, 0], self.centroids[:, 1], 
                          c='red', s=50, alpha=0.7, marker='x')
        axes[0, 0].set_title(f'Alpha Granules Detected (n={len(self.centroids)})')
        axes[0, 0].axis('off')
        
        # Clustering visualization
        if 'clustering_results' in self.activation_metrics:
            labels = self.activation_metrics['clustering_results']['labels']
            unique_labels = set(labels)
            colors = plt.cm.Spectral(np.linspace(0, 1, len(unique_labels)))
            
            axes[0, 1].imshow(self.image_rgb, alpha=0.3)
            for k, col in zip(unique_labels, colors):
                if k == -1:
                    col = [0, 0, 0, 1]  # Black for noise
                
                class_member_mask = (labels == k)
                xy = self.centroids[class_member_mask]
                axes[0, 1].scatter(xy[:, 0], xy[:, 1], c=[col], s=100, alpha=0.8)
            
            axes[0, 1].set_title(f'DBSCAN Clustering\n'
                               f'{self.activation_metrics["clustering_results"]["n_clusters"]} clusters')
            axes[0, 1].axis('off')
        
        # Distance distribution
        if 'distance_metrics' in self.activation_metrics:
            nearest_distances = self.activation_metrics['distance_metrics']['nearest_distances']
            axes[0, 2].hist(nearest_distances, bins=20, alpha=0.7, color='skyblue', edgecolor='black')
            axes[0, 2].axvline(np.mean(nearest_distances), color='red', linestyle='--', 
                              label=f'Mean: {np.mean(nearest_distances):.1f}')
            axes[0, 2].set_title('Nearest Neighbor Distances')
            axes[0, 2].set_xlabel('Distance (pixels)')
            axes[0, 2].set_ylabel('Frequency')
            axes[0, 2].legend()
        
        # Activation metrics summary
        axes[1, 0].axis('off')
        if hasattr(self, 'activation_metrics') and self.activation_metrics:
            metrics_text = f"""
CLASSIFICATION: {self.activation_metrics['classification']}
Confidence: {self.activation_metrics['confidence']:.2f}
Activation Score: {self.activation_metrics['activation_score']:.2f}

Key Metrics:
• Clustering %: {self.activation_metrics['clustering_results']['clustering_percentage']:.1f}%
• Grouping %: {self.activation_metrics['proximity_results']['grouping_percentage']:.1f}%
• Avg Neighbors: {self.activation_metrics['proximity_results']['avg_neighbors']:.1f}
• Clark-Evans Index: {self.activation_metrics['spatial_results']['clark_evans_index']:.2f}

Criteria Met:
{chr(10).join(['• ' + criterion for criterion in self.activation_metrics['criteria_met']])}
            """
            axes[1, 0].text(0.05, 0.95, metrics_text, transform=axes[1, 0].transAxes, 
                           fontsize=10, verticalalignment='top', fontfamily='monospace',
                           bbox=dict(boxstyle="round,pad=0.3", facecolor="lightgray"))
        
        # Cluster size distribution
        if 'clustering_results' in self.activation_metrics:
            cluster_sizes = self.activation_metrics['clustering_results']['cluster_sizes']
            if cluster_sizes:
                axes[1, 1].bar(range(len(cluster_sizes)), cluster_sizes, color='lightcoral')
                axes[1, 1].set_title('Cluster Size Distribution')
                axes[1, 1].set_xlabel('Cluster ID')
                axes[1, 1].set_ylabel('Number of Granules')
            else:
                axes[1, 1].text(0.5, 0.5, 'No clusters detected', 
                               ha='center', va='center', transform=axes[1, 1].transAxes)
                axes[1, 1].set_title('Cluster Size Distribution')
        
        # Neighbor count distribution
        if 'proximity_results' in self.activation_metrics:
            neighbor_counts = self.activation_metrics['proximity_results']['neighbor_counts']
            axes[1, 2].hist(neighbor_counts, bins=range(max(neighbor_counts) + 2), 
                           alpha=0.7, color='lightgreen', edgecolor='black')
            axes[1, 2].set_title('Neighbor Count Distribution')
            axes[1, 2].set_xlabel('Number of Neighbors')
            axes[1, 2].set_ylabel('Frequency')
        
        plt.tight_layout()
        
        if save_path:
            plt.savefig(save_path, dpi=300, bbox_inches='tight')
            print(f"Visualization saved to: {save_path}")
        
        plt.show()
    
    def generate_report(self):
        """Generate a comprehensive text report"""
        if not hasattr(self, 'activation_metrics') or not self.activation_metrics:
            self.classify_activation_state()
        
        report = f"""
========================================
ALPHA GRANULE ACTIVATION ANALYSIS REPORT
========================================

CLASSIFICATION: {self.activation_metrics['classification']}
Confidence Level: {self.activation_metrics['confidence']:.2f} ({self.activation_metrics['confidence']*100:.1f}%)
Overall Activation Score: {self.activation_metrics['activation_score']:.3f}

GRANULE COUNT: {self.activation_metrics['n_granules']}

CLUSTERING ANALYSIS:
• Number of clusters: {self.activation_metrics['clustering_results']['n_clusters']}
• Granules in clusters: {self.activation_metrics['clustering_results']['clustered_granules']} ({self.activation_metrics['clustering_results']['clustering_percentage']:.1f}%)
• Isolated granules: {self.activation_metrics['clustering_results']['n_noise']}
• Average cluster size: {np.mean(self.activation_metrics['clustering_results']['cluster_sizes']) if self.activation_metrics['clustering_results']['cluster_sizes'] else 0:.1f}

PROXIMITY ANALYSIS:
• Granules with neighbors: {self.activation_metrics['proximity_results']['grouped_granules']} ({self.activation_metrics['proximity_results']['grouping_percentage']:.1f}%)
• Isolated granules: {self.activation_metrics['proximity_results']['isolated_granules']}
• Average neighbors per granule: {self.activation_metrics['proximity_results']['avg_neighbors']:.2f}
• Proximity threshold used: {self.activation_metrics['proximity_results']['proximity_threshold']:.1f} pixels

SPATIAL DISTRIBUTION:
• Clark-Evans Index: {self.activation_metrics['spatial_results']['clark_evans_index']:.3f}
  ({'Clustered' if self.activation_metrics['spatial_results']['clark_evans_index'] < 1.0 else 'Random/Dispersed'})
• Spatial distribution score: {self.activation_metrics['spatial_results']['spatial_distribution_score']:.3f}

DISTANCE METRICS:
• Average nearest neighbor distance: {self.activation_metrics['distance_metrics']['avg_nearest_distance']:.1f} pixels
• Mean all pairwise distances: {self.activation_metrics['distance_metrics']['mean_all_distances']:.1f} pixels

ACTIVATION CRITERIA MET:
{chr(10).join(['• ' + criterion for criterion in self.activation_metrics['criteria_met']])}

INTERPRETATION:
        """
        
        if self.activation_metrics['classification'] == 'ACTIVATED':
            report += """
The alpha granules show ACTIVATED state characteristics:
- High degree of clustering/grouping
- Granules are concentrated in specific areas
- Low spatial dispersion
- Multiple granules in close proximity
This suggests the platelet is in an activated state, likely in response to stimulation.
            """
        elif self.activation_metrics['classification'] == 'PARTIALLY_ACTIVATED':
            report += """
The alpha granules show PARTIALLY ACTIVATED state characteristics:
- Moderate clustering with some dispersed granules
- Mixed pattern of grouped and isolated granules
- Intermediate spatial organization
This suggests the platelet may be in early stages of activation or partially stimulated.
            """
        else:
            report += """
The alpha granules show UNACTIVATED state characteristics:
- Low degree of clustering
- Granules are more uniformly distributed
- High spatial dispersion
- Most granules are isolated
This suggests the platelet is in a resting, unactivated state.
            """
        
        report += "\n========================================\n"
        
        return report

# Usage example
def analyze_platelet_activation(image_path, visualization_save_path=None):
    """
    Complete analysis workflow for platelet activation state
    
    Args:
        image_path: Path to annotated platelet image
        visualization_save_path: Optional path to save visualization
    
    Returns:
        Dictionary containing analysis results
    """
    # Initialize analyzer
    analyzer = AlphaGranuleActivationAnalyzer(image_path)
    
    # Extract alpha granules
    n_granules = analyzer.extract_alpha_granules()
    
    if n_granules < 3:
        print("Warning: Too few alpha granules detected for reliable analysis")
        return {"error": "Insufficient granules for analysis"}
    
    # Perform classification
    results = analyzer.classify_activation_state()
    
    # Generate visualization
    analyzer.visualize_analysis(visualization_save_path)
    
    # Print report
    print(analyzer.generate_report())
    
    return results

# Example usage:
if __name__ == "__main__":
    # Replace with your image path
    image_path = "/Users/souravhaldkar/Desktop/Platelet_codes/trial_annotated/387b_61_final.png"
    
    # Analyze activation state
    results = analyze_platelet_activation(
        image_path=image_path,
        visualization_save_path="activation_analysis_results.png"
        
    )
    
    print(f"\nFinal Classification: {results.get('classification', 'ERROR')}")
    print(f"Confidence: {results.get('confidence', 0):.2f}")

ModuleNotFoundError: No module named 'seaborn'